Import all libraries

In [7]:
import pandas as pd 
import numpy as np 
import random 

In [8]:
df=pd.read_csv('naive_bayes_large_dataset.csv')
df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,target
0,-1.677834,4.124498,1.903624,0.313286,-5.343350,-2.747603,-3.824291,2.551446,0.699270,1.539799,0
1,-1.767064,2.948188,1.861296,1.422092,-1.574938,-3.152812,-4.330113,3.286208,0.573797,2.771043,0
2,-1.042548,4.044066,1.575623,0.868728,-3.828637,-3.444082,-4.586344,3.945446,0.089368,2.876365,0
3,0.769716,5.376844,2.479242,1.629047,-4.175046,-4.593663,-3.855436,3.269665,0.029943,1.439116,0
4,-2.078334,4.605090,2.744082,1.633410,-2.788192,-2.889954,-4.570108,3.207488,-0.087710,2.881943,0


In [9]:
df.shape

(99999, 11)

Train Test split

In [6]:
def train_test(x,y,test_size=0.2,random_state=1):
    data=list(zip(x,y))
    random.seed(random_state)
    random.shuffle(data)
    x_s,y_s=zip(*data)
    
    split=int(len(x_s)*(1-test_size))
    x_tr=x_s[:split]
    x_te=x_s[split:]
    y_tr=y_s[:split]
    y_te=y_s[split:]
    return np.array(x_tr),np.array(x_te),np.array(y_tr),np.array(y_te)

x=df.drop(columns=['target']).values
y=df['target'].values
x_train,x_test,y_train,y_test=train_test(x,y,test_size=0.2,random_state=2)

Implementing Logistic Regression from scratch

In [10]:
class NaiveBayes:
    def __init__(self):
        self.classes = None
        self.mean = {}
        self.var = {}
        self.priors = {}

    def fit(self, X, y):
        self.classes = np.unique(y)

        for c in self.classes:
            X_c = X[y == c]
            self.mean[c] = np.mean(X_c, axis=0)
            self.var[c] = np.var(X_c, axis=0)
            self.priors[c] = X_c.shape[0] / X.shape[0]

    def gaussian_pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        numerator = np.exp(- (x - mean) ** 2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def predict(self, X):
        predictions = []
        
        for x in X:
            posteriors = []

            for c in self.classes:
                prior = np.log(self.priors[c])
                conditional = np.sum(np.log(self.gaussian_pdf(c, x)))
                posterior = prior + conditional
                posteriors.append(posterior)

            predictions.append(self.classes[np.argmax(posteriors)])

        return np.array(predictions)

In [13]:
lr=NaiveBayes()
lr.fit(x_train,y_train)
y_predict=lr.predict(x_test)
y_predict

array([1, 1, 1, ..., 0, 0, 0], shape=(20000,))

Accuracy of the model

In [12]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

print("Accuracy is:", accuracy(y_test, y_predict)*100)

Accuracy is: 100.0
